# 🧪 W6-D4 概念实验：Agent 推理评测与 Prompt 工程

> 配套阅读：`第6周-Day4-Agent推理与Prompt工程.md`（三大基准、四大策略、RTF 框架在那边）
>
> 本 notebook 用可运行实验回答四个问题：
> 1. **pass@k 怎么算才无偏？** —— 亲手实现 OpenAI 官方公式并和朴素算法对比
> 2. **四种 Prompt 策略**（直接 / 少样本 / CoT / RTF）在"准确率 × 格式可解析率"上差多少？
> 3. **RTF 框架**（Role-Task-Format）：约束输出格式的 prompt 长什么样？解析成功率提升多大？
> 4. **模型级别雷达**：小/中/大模型差在哪几维？

环境：仅 numpy / 标准库 / matplotlib，全部本地模拟。

## 实验 1：pass@k 无偏估计（HumanEval 同款公式）

对一道题采样 n 次、对了 c 次，估计 pass@k。**朴素算法**（重复实验数频率）在单题上有偏且抖动大；
**官方无偏估计** E[pass@k] = 1 − C(n−c, k)/C(n, k) 只用一次采样的统计量就给出无偏结果。

In [ ]:
import numpy as np
from math import comb
rng = np.random.default_rng(42)

def pass_at_k_unbiased(n, c, k):
    """官方公式：1 - C(n-c,k)/C(n,k)。c>=n-k+1 时必为 1。"""
    if n - c < k:
        return 1.0
    prod = 1.0
    for i in range(k):                     # 数值稳定的连乘，避免大组合数溢出
        prod *= (n - c - i) / (n - i)
    return 1.0 - prod

p_true = 0.30                    # 单次采样通过率（真值只有出题人知道）
n = 10                           # 对每道题采样 10 次、数对了 c 次

# (a) 单次观测的估计：会抖 —— 观测到的 c 和真值有抽样误差
obs = (rng.random(n) < p_true)
c_obs = int(obs.sum())
print(f"真值 p={p_true}；本次采样 n={n} 观测到 c={c_obs}（抽样本身就带波动）")
print(f"{'k':>3} {'理论 1-(1-p)^k':>14} {'本次观测的无偏估计':>16}")
for k in [1, 2, 5, 10]:
    print(f"{k:>3} {1 - (1 - p_true) ** k:>14.1%} {pass_at_k_unbiased(n, c_obs, k):>16.1%}")

# (b) 无偏性的真正含义：对大量观测取平均，估计值收敛到理论值
print(f"\n对 3000 次独立观测取平均（每次重新观测 c）：")
print(f"{'k':>3} {'理论值':>8} {'无偏公式均值':>12}")
for k in [1, 2, 5, 10]:
    ests = []
    for _ in range(3000):
        c = int((rng.random(n) < p_true).sum())
        ests.append(pass_at_k_unbiased(n, c, k))
    print(f"{k:>3} {1 - (1 - p_true) ** k:>8.1%} {np.mean(ests):>12.1%}")
print("\n解读：单次观测的估计会抖，但无偏公式平均意义=真值；")
print("且每次只需 n 次采样。评测集=164 道 HumanEval 题各算一次 pass@k 再取平均。")

## 实验 2：四种 Prompt 策略 —— 准确率 × 可解析率

工程现实：**答案再对，格式解析失败 = 0 分**。
用两个独立概率建模每种策略：`正确率` 与 `格式可解析率`，跑 500 道"模拟评测题"，
统计最终可用率 = 可解析 且 正确。

In [ ]:
rng = np.random.default_rng(7)

STRATEGIES = {
    #                正确率  可解析率   相对成本
    "直接问":        (0.58, 0.72, 1.0),
    "少样本示例":    (0.70, 0.80, 2.5),
    "CoT 分步":      (0.78, 0.68, 4.0),   # 过程自由发挥 → 格式更飘
    "RTF+CoT+schema":(0.79, 0.97, 4.2),   # RTF 锁定输出格式
}
n_q = 500
print(f"{'策略':<16}{'正确率':>8}{'可解析率':>9}{'可用率':>8}{'相对成本':>9}")
results = {}
for name, (p_corr, p_parse, cost) in STRATEGIES.items():
    correct = rng.random(n_q) < p_corr
    parseable = rng.random(n_q) < p_parse
    usable = (correct & parseable).mean()
    results[name] = (correct.mean(), parseable.mean(), usable, cost)
    print(f"{name:<16}{correct.mean():>8.1%}{parseable.mean():>9.1%}{usable:>8.1%}{cost:>8.1f}x")
best = max(results, key=lambda x: results[x][2])
print(f"\n可用率最高：{best} —— CoT 的'对但格式乱'被 RTF 的格式约束救回来了。")
print("教训：评测要同时报三个数（正确/可解析/可用），只报正确率会高估系统。")

## 实验 3：RTF 框架 —— Role / Task / Format 模板组装器

把 md 里的 RTF 黄金法则写成可复用的组装函数（真实项目里这就是 prompt 模板引擎的雏形），
再模拟对比：自由文本输出 vs RTF 指定 JSON schema 的**下游解析成功率**。

In [ ]:
import json, re

def build_rtf_prompt(role, task, format_spec, context=""):
    return (f"# 角色\n你是{role}。\n\n# 任务\n{task}\n\n"
            f"# 上下文\n{context}\n\n# 输出格式（严格遵守，不要输出任何其他内容）\n{format_spec}")

schema = '{"items": [{"name": "商品名", "stock": 42}], "advice": "一句话建议"}'
prompt = build_rtf_prompt(
    role="糖水店库存分析 Agent",
    task="根据工具返回的库存数据，找出需要补货的商品并给出建议",
    format_spec=schema,
    context='工具结果：{"杨枝甘露": 42, "芒果西米露": 3, "桂花酸梅汤": 0}')
print(prompt)

# --- 下游解析实验：自由文本 vs RTF-JSON 的字段级解析 ---
PRODUCTS = ["杨枝甘露", "芒果西米露", "桂花酸梅汤"]
def gen_free():                          # 模拟自由文本：经常缺字段/换说法
    p = PRODUCTS[rng.integers(3)]
    r = rng.random()
    if r < 0.35: return f"{p}卖得不错，放心。"                    # 缺库存数
    if r < 0.50: return f"{p}库存只剩 {rng.integers(0, 8)} 杯。"   # 缺补货建议
    return f"{p}库存 {rng.integers(0, 50)} 杯，建议尽快补货。"      # 完整

def parse_free(o):                      # 下游要的三个字段：商品/库存数/建议
    m = re.search(r"([\u4e00-\u9fff]{2,6})库存.{0,4}?(\d+)", o)
    return bool(m) and "建议" in o

def parse_json(o):
    try:
        d = json.loads(o)
        return "items" in d and "advice" in d
    except Exception:
        return False

n = 3000
rtf_fail = 0.03                          # RTF 偶发夹带说明文字导致解析失败
ok_free = sum(parse_free(gen_free()) for _ in range(n)) / n
ok_json = sum(parse_json('{"items":[{"name":"x","stock":3}],"advice":"补货"}'
                        if rng.random() > rtf_fail else "好的，分析如下：...") for _ in range(n)) / n
print(f"\n字段级解析成功率：自由文本 {ok_free:.0%} vs RTF-JSON {ok_json:.0%}")
print("自由文本'看起来能读'≠程序能稳定抽出字段；格式约束是把 LLM 接进系统的第一道工程化。")

## 实验 4：可视化 —— 模型级别雷达 + pass@k 曲线

In [ ]:
# matplotlib 中文字体配置（NotoSansCJK，每次画图前先跑这段）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = fontManager_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

# 左：三个级别模型的能力雷达（模拟评测数据）
dims = ["MMLU 通识", "GSM8K 数学", "HumanEval 代码", "BBH 推理", "格式遵循", "指令遵循"]
models = {"7B 小模型": [45, 25, 20, 35, 55, 60],
          "32B 中模型": [62, 55, 45, 52, 75, 78],
          "旗舰模型": [82, 84, 75, 78, 92, 90]}
angles = np.linspace(0, 2 * np.pi, len(dims), endpoint=False).tolist(); angles += angles[:1]
for name, vals, color in zip(models, models.values(), ["#adb5bd", "#219ebc", "#fb8500"]):
    v = vals + vals[:1]
    axes[0].plot(angles, v, "o-", color=color, label=name, markersize=4)
axes[0].set_xticks(angles[:-1]); axes[0].set_xticklabels(dims, fontsize=8)
axes[0].set_title("模型级别能力画像（模拟）"); axes[0].legend(fontsize=9, loc="lower right")

# 右：pass@k 随 k 上升（不同单次通过率 p）
ks = np.arange(1, 11)
for p, color in [(0.1, "#adb5bd"), (0.3, "#219ebc"), (0.6, "#fb8500")]:
    axes[1].plot(ks, 1 - (1 - p) ** ks, "o-", color=color, label=f"单次通过率 p={p}")
axes[1].set_xlabel("k（采样次数）"); axes[1].set_ylabel("pass@k")
axes[1].set_title("采样预算换通过率：弱模型也能靠 k 堆上去（成本×k）")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("右图解读：p=0.3 的模型 pass@5≈84% —— 但成本也×5；")
print("=> '模型升级' vs '多采样' 是一条成本曲线上的两种走法（自洽性就是多采样的特例）。")

## 小结

- pass@k 有无偏公式，评测要报"正确率/可解析率/可用率"三个数
- Prompt 策略=准确率与格式约束的组合拳：RTF+CoT+schema 是工程默认起手式
- 模型选型与采样预算可以互相替换，本质都是成本-效果权衡
- 下一步：D5 框架设计（把 prompt/评测沉淀成可维护的工程组件）